# GPR–FDTD–FWI: A Plain-English Project Update

*An end-to-end status note — where we started, what we have accomplished, where
we are right now, and where we are going. Written to be read by anyone, with
every abbreviation and concept spelled out.*

**Author:** Claude (Opus 4.8)  ·  **Compiled:** 2026-06-01
**Scope:** the whole repository — ~276 numbered experiments, 47 written
experiment logs, and five guiding research papers.

---

### How to read this note

This is a *story*, not a run log. Each section answers one question in ordinary
language first, and only then shows the evidence (a figure, an animation, or a
small table). If you only read the **bold one-liners** and look at the figures,
you will still get the whole arc.

The figures and animations are pulled directly from the experiment output
folders (`outputs/experiments/NNN_.../figures/`). They are linked by relative
path, so they display when this notebook is opened inside the repository.


## 0. The 60-second summary

**What this is.** A from-scratch computer model that simulates how
ground-penetrating radar (GPR) "sees" steel reinforcing bars (rebar) buried in
concrete, plus an automatic procedure that runs that model *in reverse* to
recover where each bar is and how thick it is — from the radar echoes alone.

**Where it started.** As a focused technical assessment: build the radar
simulator, build the reverse-solver, make it run on a GPU, and demonstrate it
recovering three rebars in a synthetic concrete slab. That part works and is
done.

**What it became.** A sustained research campaign around one stubborn question:
*we can always find **where** a bar is, but can we trust **how thick** we say it
is?* Bar thickness (radius) turns out to be the hard, scientifically interesting
quantity. Roughly 276 experiments later, we have mapped exactly when radius is
trustworthy, when it is only an interval, and which physical levers move it.

**Where we are now.** A confidence-aware 2-D pipeline that: detects bars from a
raw radar image, refines each one with the simulator, **reports an honest
uncertainty interval instead of a single over-confident number**, and handles
realistic nuisances (unknown radar pulse shape, 10% noise, several bars close
together, bars of different sizes).

**Where we are going.** Treat the *measurement design itself* (radar bandwidth,
antenna spacing, number of scan positions) as a first-class tuning knob, package
the multi-stage pipeline into one reproducible command, and then move from
synthetic data toward lab/field data.


## 1. The real-world problem, in plain words

Reinforced concrete is concrete with steel bars (**rebar**) inside it for
strength. Engineers often need to know, *without breaking the concrete open*:

- **Where** are the bars (how deep, and where side-to-side)?
- **How thick** is each bar (its radius / diameter)? Thinning from corrosion is
  a safety issue.

The standard non-destructive tool is **GPR — Ground-Penetrating Radar**. You
drag a small radar antenna across the surface. It repeatedly sends a short
electromagnetic pulse into the concrete and records the echoes. A buried bar
reflects the pulse, and as the antenna passes over it the echo arrives earliest
right above the bar and later to either side — tracing out a hyperbola (an arch
shape) in the recorded image.

That recorded image — echo strength versus *antenna position* (horizontal) and
*time* (vertical) — is called a **B-scan** (a "brightness scan", the standard
radar/ultrasound term for a 2-D position-vs-time image).

**The catch.** A B-scan tells you a bar is *there* fairly easily. Pulling out an
accurate *size* is much harder, because a slightly bigger bar and a slightly
deeper/differently-surrounded bar can produce almost the same echoes. Untangling
that is the scientific heart of this project.


## 2. The toolchain we built — the "engine room"

Everything rests on two pieces of machinery and a speed layer. Here they are in
plain terms.

### 2a. Forward simulation — "predict the echoes" (FDTD)

**FDTD = Finite-Difference Time-Domain.** It is a standard, physically faithful
way to simulate electromagnetic waves. You chop space into a fine grid of tiny
cells and step time forward in tiny ticks; at each tick you update the electric
and magnetic fields in each cell from its neighbours, following Maxwell's
equations (the laws of electromagnetism). Do this for enough ticks and you have
simulated a radar pulse travelling into the concrete, bouncing off the steel,
and coming back.

A few terms you will see:
- **Yee grid / leapfrog** — the specific, decades-old recipe for arranging the
  electric and magnetic fields and updating them in alternation. It is the
  textbook-correct way to do FDTD.
- **TMz polarization (Ez, Hx, Hy)** — we simulate a 2-D slice of the world. This
  just names which field components live in that slice.
- **CPML (absorbing boundary)** — the simulation grid has edges, but the real
  concrete does not. CPML is a special "soft wall" at the grid edge that soaks
  up outgoing waves so they do not bounce back and pollute the result, faking an
  infinite medium.

The simulator was validated against known physics (wave speed correct to ~2%,
stable runs, energy decays properly).


<img src="../../outputs/figures/ground_truth.png" width="560"><br>
<em><b>Figure.</b> The synthetic test world: a concrete slab (with a thin air gap on top) holding three steel rebars. This is the 'ground truth' the simulator knows and the inversion must rediscover.</em>

<img src="../../outputs/figures/bscan.png" width="560"><br>
<em><b>Figure.</b> The simulated B-scan (radar image). Horizontal axis = antenna position; vertical axis = echo travel-time (deeper = lower). Each rebar produces a clear hyperbola (arch). This is the only thing the reverse-solver is allowed to look at.</em>

<img src="../../outputs/animations/wave_propagation.gif" width="560"><br>
<em><b>Figure.</b> The forward simulation in motion: a single radar pulse spreading into the slab and scattering off the three bars. Watching the wavefield like this is how we sanity-check that the physics engine behaves correctly.</em>

### 2b. Reverse solving — "echoes ➜ model" (FWI by the adjoint method)

**FWI = Full-Waveform Inversion.** "Inversion" means going backwards: instead of
*model ➜ predicted echoes*, we want *measured echoes ➜ model*. "Full-waveform"
means we try to match the **entire wiggle** of every recorded trace, not just
the arrival time of a blip. That uses far more of the information in the data.

The recipe is a guess-and-correct loop:
1. Start from a guess model (e.g. plain concrete, no bars).
2. Simulate its B-scan with the FDTD engine.
3. Compare to the measured B-scan — the mismatch is the **misfit** (also called
   the **objective**; lower is better).
4. Nudge the model to reduce the misfit, and repeat.

The clever part is step 4. The model has thousands of unknowns, so we cannot try
them one by one. The **adjoint-state method** computes how the misfit would
change for *every* model cell at once, using only **two** simulations per
iteration: the normal forward run, plus one extra run where the *leftover
mismatch* is injected back into the medium and propagated **backwards** in time.
Multiplying the forward and back-propagated fields together yields the
**gradient** — the direction to push the model. It is the same mathematics that
powers "backpropagation" in neural networks, applied to a wave equation.


<img src="../../outputs/experiments/052_wavelet_mismatch_radius_exact/figures/amp_high10_nominal_r7p0_z91_muted_residual_backprop.gif" width="560"><br>
<em><b>Figure.</b> The adjoint method made visible: the leftover data mismatch is injected and propagated backwards through the slab. Where this back-propagated field overlaps the forward field, the algorithm learns how to correct the model. Two simulations per step give the update for the whole grid.</em>

<img src="../../outputs/figures/inversion_comparison.png" width="720"><br>
<em><b>Figure.</b> A full inversion result: starting guess vs. recovered model vs. truth. From the radar echoes alone, the reverse-solver places the bars correctly.</em>

<img src="../../outputs/figures/convergence.png" width="520"><br>
<em><b>Figure.</b> Convergence curve: the misfit (data mismatch) dropping iteration by iteration as the model is corrected. A steadily falling curve is the sign of a healthy inversion.</em>

### 2c. The speed layer — GPU acceleration

Each FDTD simulation is millions of tiny, identical cell-updates, which is
exactly what a **GPU (graphics processor)** is built to do quickly. The engine
has a GPU version so the many simulations an inversion needs become practical.
This work moved from a laptop to an NVIDIA **DGX Spark** machine, and from that
point on **every simulation has been run on the GPU**.


<img src="../../outputs/figures/gpu_scaling.png" width="520"><br>
<em><b>Figure.</b> GPU vs. CPU timing for the forward simulation. The GPU speed-up is what makes a 276-experiment campaign — many of which sweep dozens of candidate models each — feasible at all.</em>

## 3. Where we started: the baseline single-rebar pipeline
*(experiments 001–023)*

With the engine working, the first research milestone was to nail the simplest
honest test: **one** steel bar in concrete, synthetic data, recover its
horizontal position *x*, its depth *z*, and its radius *r*.

**Result: on clean synthetic data this works perfectly** — recovered
*x* = 250 mm, *z* ≈ 90 mm, *r* = 6.0 mm, with the recovered model matching truth
exactly (model error "NRMS" = 0; **NRMS = Normalized Root-Mean-Square error**, a
standard 0-is-perfect mismatch score). It stays correct under moderate added
noise.

The working recipe was a **coarse ➜ fine ➜ polish** search:
> a quick search on a coarse 2 mm grid to get into the right neighbourhood, then
> a 1 mm local refinement, then a final 1 mm "grid polish" that pins the radius.

**But one difficulty surfaced that drives the entire rest of the project:**
position is easy, **radius is delicate**. A continuous optimizer (Powell's
method) kept sliding toward a *too-thick* bar (~6.9–7.0 mm instead of 6.0 mm),
and only the careful grid-polish step recovered the true size. Part of this is
**grid rasterization**: when you draw a small circle on a 1 mm grid, two nearby
radii (say 4.0 and 4.1 mm) can fill *exactly the same cells*, so the physics
literally cannot tell them apart. Radius identifiability — and how to report it
honestly — becomes the through-line of the whole campaign.


## 4. The central question, tested against five papers
*(experiments 024–058)*

We had a pipeline that nailed the *clean* case. So we asked the sharp question:

> **Which assumption, if broken, would actually move the radius answer?**

To avoid guessing, we used **five published FWI research papers as a menu of
candidate ideas**, and tested each one on our own radius problem *before*
trusting it. Here is the menu in plain English:

| # | Paper (short name) | Its core idea | What we did with it |
|---|---|---|---|
| 1 | **WRI** 2022 (Wavefield Reconstruction Inversion) | Start with low radar frequencies, add higher ones gradually; don't let high frequencies dominate early | Borrowed the *frequency-weighting idea*; did **not** build its heavy solver |
| 2 | **PEBDD** 2021 (Progressively Expanded Bandwidth) | Fit a narrow low-frequency band first, then widen it in stages | Built a staged-bandwidth runner + a spectrum-design tool |
| 3 | **OT-LS** 2025 (Optimal Transport then Least-Squares) | Use a "transport" distance to avoid a classic trap (cycle-skipping), then switch to ordinary fitting | Added the paper's trap-detector as a *diagnostic* |
| 4 | **Quadratic-W2** 2024 (Wasserstein distance for GPR) | Replace ordinary fitting with an optimal-transport distance for noise robustness | Built it, tested it on our problem |
| 5 | **IFWI** 2025 (Implicit/neural multi-parameter) | Let a small neural network represent the material map | Wrote a feasibility note; **deferred** |

Two pieces of jargon those papers revolve around:
- **Cycle-skipping** — the classic FWI failure where the simulation lines a
  wave wiggle up with the *wrong* neighbouring wiggle (one cycle off) and then
  confidently optimizes toward a wrong answer.
- **Optimal transport / Wasserstein (W2)** — a way of measuring the difference
  between two signals by how much "work" it takes to slide one into the other,
  rather than subtracting them point-by-point. Good against timing shifts.

Each branch below became a clean **yes/no finding about our specific problem.**


### 4.1 Is the radius trouble just "cycle-skipping"? — **No.**
*(OT-LS paper · experiments 024–026)*

If the too-thick-bar trap were cycle-skipping, the OT-LS paper's trap-detector
(a trace-shift score called NRCCC) would flag it. We computed that score for
every candidate, including the wrong-radius ones.

**Finding:** the detector reads "perfectly safe" (score = 1.0) for *every*
candidate, including the wrong ones. So the radius problem is **not** a timing /
cycle-skipping problem — it is an **amplitude-and-detail** problem. This ruled
the whole optimal-transport branch *out* for our case (we kept it only as a
diagnostic for future field data).


### 4.2 Does staged bandwidth fix radius? — **It builds seeds, not sizes.**
*(PEBDD paper · experiments 028–045)*

We built the staged low-to-high bandwidth procedure, and even a *spectrum-design*
tool to choose the frequency bands from the data instead of guessing.

**Finding:** the low-frequency band that the method leans on contains only about
**2%** of the information that actually distinguishes radius (the radius-telling
content lives at roughly 1.0–2.5 GHz). So staged bandwidth is a good *starting-
point builder* — it gets position into the right neighbourhood — but it does
**not** select the size by itself; the polish step still does that.


<img src="../../outputs/experiments/043_pebd_spectrum_design_exact/figures/residual_spectra.png" width="640"><br>
<em><b>Figure.</b> Spectrum-design diagnostic: how much radius-distinguishing energy lives in each frequency band. The low band the PEBDD method emphasizes is nearly empty of size information — which is why it improves position seeds but not the size answer. (Useful content sits around 1–2.5 GHz.)</em>

### 4.3 Does mixing two frequencies help? — **No, plain averaging hurts.**
*(WRI cumulative-frequency idea · experiments 038–049)*

Intuition says combining a 1.0 GHz and a 1.5 GHz view should add information.

**Finding:** the 1.0 GHz signal is ~30× *less* sensitive to radius than the
1.5 GHz signal, so **averaging them equally cuts the radius "margin" roughly in
half** (the *margin* is how clearly the best radius beats the next-best — bigger
is more confident). The lesson: don't blindly average frequencies. Use the
size-sensitive band (1.5 GHz), optionally carrying the low band at only ~25%
weight for stability.


<img src="../../outputs/experiments/046_frequency_weight_radius_margin_exact/figures/frequency_weight_radius_profiles.png" width="640"><br>
<em><b>Figure.</b> Radius-misfit curves under five frequency-weighting schemes. The deepest, cleanest dip at the true radius comes from the size-sensitive 1.5 GHz band; equal averaging (a shallower dip) dilutes the size evidence.</em>

### 4.4 Does the Wasserstein (W2) distance help? — **Right tool, wrong problem.**
*(Quadratic-W2 paper · experiments 048–051)*

We faithfully reproduced the paper's headline first: on artificially
time-shifted pulses, the W2 distance is smooth and well-behaved where ordinary
least-squares is bumpy and trap-prone. **The paper's claim holds.**

**But** when we put W2 on our actual radius problem, the radius signal *collapsed
by four orders of magnitude* (margin 1.06e-7 with W2 vs 1.04e-3 with ordinary
fitting). The reason: W2 needs a normalization step that throws away exactly the
*amplitude* detail our radius answer depends on. **W2 fixes a timing problem we
don't have, and would erase the amplitude problem we do have.** Rejected for
radius selection; kept on the shelf as a basin-finder for bad starting models.


<img src="../../outputs/experiments/048_trace_wasserstein_convexity_smoke/figures/trace_w2_convexity.png" width="640"><br>
<em><b>Figure.</b> Gate 1 (passed): on shifted pulses, ordinary least-squares (left) is bumpy and full of false minima, while the W2 distance (right) is smooth — exactly as the paper claims.</em>

<img src="../../outputs/experiments/050_w2_landscape_exact_beta8_ds16/figures/w2_radius_profiles.png" width="640"><br>
<em><b>Figure.</b> Gate 2 (failed for our use): on the real rebar-radius problem, least-squares (left) has a sharp dip at the true radius, but W2 (right, note the tiny vertical scale) is almost flat — it cannot tell sizes apart. Hence W2 is rejected as the radius objective.</em>

### 4.5 Could "wrong material" be masquerading as "wrong size"? — **No.**
*(experiment 056)*

Maybe a wrong radius could hide behind a wrong guess for the concrete's
properties or the steel's conductivity. We swept those material properties
together with radius.

**Finding:** at the correct position, the true radius with the true material
wins cleanly. The concrete property is well-identified; the steel's conductivity
saturates (any "very conductive" value looks the same, which is physically
expected). **Material does not explain the radius bias**, so we do *not* add free
material knobs to the size optimizer.


<img src="../../outputs/experiments/056_material_tradeoff_fixed_xz_exact/figures/material_profiled_radius.png" width="560"><br>
<em><b>Figure.</b> Best achievable misfit vs. radius after also optimizing the material properties at each radius. The clean minimum stays at the true radius — a wrong size cannot be 'bought back' by adjusting materials.</em>

### 4.6 The real culprit: an unknown radar pulse shape — **the big finding**
*(experiments 052–055)*

On real equipment you never know the emitted radar pulse *exactly* — its
amplitude, its precise start-time, and its centre frequency all drift. We tested
what a small such mismatch does to the size answer.

**Finding (the failure):** a *modest* pulse mismatch wrecks the radius. With the
modelled pulse held fixed and only the observed pulse perturbed:

| Pulse mismatch (observed) | Recovered radius |
|---|---|
| none (nominal) | **6.0 mm** ✓ |
| centre-frequency −10% | 5.4 mm ✗ (hits low bound) |
| centre-frequency +10% | 7.8 mm ✗ (hits high bound) |
| start-time +50 ps | 5.4 mm ✗ |
| start-time −50 ps | 7.8 mm ✗ |
| amplitude +10% | 7.0 mm ✗ |

This is the kind of failure that would only show up *after* moving to real data —
so finding it on synthetics is exactly the point.


<img src="../../outputs/experiments/052_wavelet_mismatch_radius_exact/figures/wavelet_mismatch_radius_profiles.png" width="640"><br>
<em><b>Figure.</b> Failure mode: each curve is a different small pulse mismatch. Only the nominal/amplitude cases dip at the true 6 mm radius; timing and frequency mismatches push the best radius all the way to the grid edge.</em>

**The fix — "source profiling".** Instead of pretending we know the pulse,
let the algorithm *also* solve for three small pulse nuisance parameters at each
candidate: one **amplitude** scale, one **time-shift**, and one
**centre-frequency** scale. Added one at a time, they fix the cases one at a
time; with all three, **every** mismatched case recovers the true 6.0 mm radius
again. This is the single most important new ingredient in the whole project.


<img src="../../outputs/experiments/055_wavelet_mismatch_radius_amp_time_freqfit/figures/wavelet_mismatch_radius_profiles.png" width="640"><br>
<em><b>Figure.</b> The fix: with the pulse's amplitude, timing, and frequency profiled as nuisances, all seven mismatch curves collapse back onto the same clean dip at the true 6 mm radius.</em>

### 4.7 The synthesis: a "source-profiled radius polish" runner
*(experiments 057–058)*

Every surviving idea was assembled into one reusable procedure: scan a small
local grid of candidate (*x, z, r*), and at each candidate quietly solve for the
three pulse nuisances, then report not just the best radius but the **top
candidates and the margin** (how clearly the winner won).

- **Nominal case (057):** recovers *r* = 6.0 mm cleanly.
- **Deliberately mismatched case (058):** still recovers *r* = 6.0 mm **and**
  correctly reports the injected pulse amplitude/time/frequency — proving the
  size isn't secretly absorbing the pulse error.


<img src="../../outputs/experiments/058_source_profiled_polish_combined_mismatch/figures/source_profiled_radius_profile.png" width="600"><br>
<em><b>Figure.</b> The production-style 'source-profiled polish' on a deliberately mismatched case: the true 6 mm radius wins, and the recovered pulse nuisances match what was injected. Position is exact; size is correct with a real margin.</em>

<img src="../../outputs/experiments/059_source_profiled_replication_compact/figures/true_vs_high_radius_candidate_comparison.gif" width="560"><br>
<em><b>Figure.</b> Why the size question is subtle: the true-radius bar (one wavefield) and a too-thick candidate (the other) produce echoes that look almost identical. Telling them apart is precisely what the source-profiled polish + margin reporting is built to do.</em>

**Verdict of the five-paper arc (a hierarchy of usefulness for *our* problem):**

```
Strongly useful:   source-pulse profiling (amplitude + time + frequency)
                   frequency weighting toward the size-sensitive band
                   staged bandwidth — but only as a position-seed builder

Useful as diagnostic only:   the cycle-skipping / trace-shift detector
                             the W2 distance (proves the shift claim)

Rejected for this problem:   W2 as the radius objective
                             blind multi-frequency averaging
                             free material knobs in the size optimizer
                             full WRI solver and full neural IFWI (deferred)
```


## 5. Scaling up to several bars, and learning to say "I'm not sure"
*(experiments 063–106)*

Next milestone: more than one bar in the slab (the realistic case — rebar comes
in grids). We climbed a deliberate ladder:

1. **One shared radius** for three bars — recovered cleanly.
2. **Each bar's radius separately** — still correct, but the confidence margins
   are thinner (neighbours interfere).
3. **Each bar's full position + radius locally** — correct for left, centre, and
   right bars under 10% noise and pulse mismatch, but again with thin margins.

The thin margins forced the most important *cultural* change in the project:

> **We stopped reporting a single radius number. Every result now carries a
> confidence label (strong / moderate / weak) and an "ambiguity interval" — the
> range of radii the data cannot rule out.**

Across a 24-case replication over several noise patterns, the pipeline recovered
the true position and radius in **all 24**, but honestly labelled **22 as only
"weak" confidence** and 2 as "moderate". That is not a failure — it is the
pipeline refusing to overstate precision. We then built a **reporting-first
coordinate optimizer**: it tunes one bar at a time within guarded windows, and a
**"guarded revisit"** step goes back and rescues any bar that initially fell into
the too-thick trap once its neighbours are pinned down.


<img src="../../outputs/experiments/208_coordinate_optimizer_close_spacing_broad_ambiguity_revisit_seed13/figures/coordinate_confidence_margins.png" width="640"><br>
<em><b>Figure.</b> Multi-bar confidence reporting: per-target radius margins with strong / moderate / weak labels. The point estimates are correct, but the figure's job is to show *how confident* each one is — uncertainty is now a first-class output, not an afterthought.</em>

## 6. From "we already know where it is" to "find it from scratch"
*(experiments 107–201)*

Up to here, the refinement was *given* roughly where to look. The next milestone
removed that crutch and built a true front end: **start from the raw B-scan, with
no idea where the bars are.**

**6a. A detector.** It cleans the B-scan, turns it into an "energy" image, and
scores how well a bar at each candidate position would explain the hyperbola
arcs. One subtlety it taught us: you must account for the pulse's start-time
delay, or every bar looks too deep. On a 48-scenario sweep (depths 70–130 mm,
radii 4–10 mm, up to 10% noise — *and* again under pulse mismatch), the detector
found the bar **48 out of 48** times.


<img src="../../outputs/experiments/114_detection_multi_rebar_close_spacing_source_mismatch_noise10/figures/detection_overlay.png" width="640"><br>
<em><b>Figure.</b> The detector working on three closely-spaced bars under 10% noise and pulse mismatch, with no prior knowledge of their locations. Marked peaks are the detected seed windows it hands to the refinement stage.</em>

**6b. A cheap two-stage refinement.** Running the full fine search on every
detector hit is too slow, so we use a **coarse screen ➜ narrow polish**: a quick
2 mm screen throws away bad windows, and the expensive 1 mm source-profiled
polish runs only on survivors. This **cut runtime ~55%** (≈1060 s ➜ ≈479 s) with
the *same* final answer. A final optional **"guarded polish"** stage does a
high-quality local check.

**6c. The honest limit we mapped — shallow, thin bars.** A shallow 4 mm bar
under pulse mismatch + 10% noise is the hardest case. The pipeline gets its
position and radius **exactly right across multiple noise patterns**, but the
size margin is genuinely thin: the data support a small *interval* (about
3.9–4.0 mm), not a razor-sharp number. We chased this down carefully and proved
part of it is unavoidable **grid rasterization** (4.0 and 4.1 mm can fill
identical grid cells), which sub-cell geometry and a finer grid partly relieve.
The right scientific answer is to **report the interval.**

**6d. The strongest new lever — higher radar bandwidth.** Mild post-hoc tricks
(frequency reweighting) barely helped, but *acquiring* a higher-frequency band
(2.5 GHz) genuinely sharpened size confidence: for the deep 8 mm bar it
**collapsed the uncertainty interval to a single value** across three noise
patterns, and it strengthened the shallow 4 mm case too. This points to a
**progressive-bandwidth pipeline**: low/mid frequencies for *finding and
locating*, a dedicated high-frequency band for *final size confidence*. (And we
keep the bands explicitly separated so we never imply the low-frequency data
secretly contained high-frequency information.)


<img src="../../outputs/experiments/201_packaged_highband_material_uncertainty_r4_r8_178_200/figures/two_stage_stage_confidence_summary.png" width="680"><br>
<em><b>Figure.</b> Per-stage confidence across the packaged detector→FWI pipeline for the shallow 4 mm and deep 8 mm bars. The high-frequency stage upgrades size confidence (decisively for the 8 mm bar) while keeping the point estimates exact — and weak cases stay honestly labelled.</em>

## 7. Where we are right now: different-sized bars, close together
*(experiments 202–276 — the current frontier)*

The active research line is the realistic hard case: **three bars of *different*
sizes (5, 6, 8 mm) packed close together.** This stresses everything at once —
the bars interfere, and a wrong size on one can hide a wrong position on another.

The key discovery here is about **ordering**. A naive "tune every bar's
everything at once" search fails — it inflates the small bars. We diagnosed why
with a clean oracle test:

- With each bar's *position fixed to truth*, a joint search over the size-triple
  **correctly ranks the true (5, 6, 8) first.** So the size objective is sound.
- With positions fixed to the *detector's* (slightly off) seeds, the same search
  **fails** and prefers inflated sizes.

**Conclusion: positions must be corrected *before* sizes are estimated.** That
gave the current **staged policy**, which mirrors the papers' "low-dimensional
updates first" philosophy:

```
1. Detect bars in the B-scan                  (find candidates)
2. Assign one clean seed per physical bar     (reject duplicate/ghost hits)
3. Location-only pass, radii held fixed        (lock x/z without size cheating)
4. Focused position+size polish on hard bars   (rescue the deep/large one)
5. Joint radius estimation at the fixed grid    (read off the size-triple)
```

This recovers the true sizes across multiple noise patterns. What remains is a
narrow, honestly-reported **sideways (lateral-x) ambiguity** when bars are very
close: sometimes the data cannot distinguish 309 vs 310 mm for a bar.


<img src="../../outputs/experiments/222_joint_radius_variable_radius_true_xz_seed13/figures/joint_radius_top_candidates.png" width="600"><br>
<em><b>Figure.</b> Oracle test (positions = truth): the joint size search ranks the true (5, 6, 8) mm triple first. The size objective itself is healthy.</em>

<img src="../../outputs/experiments/223_joint_radius_variable_radius_assigned_xz_seed13/figures/joint_radius_top_candidates.png" width="600"><br>
<em><b>Figure.</b> Same search but at the detector's slightly-off positions: it now prefers inflated sizes and misses the truth. This is the evidence that *position must be fixed before size* — the reason the pipeline is staged.</em>

**The lateral ambiguity, and how acquisition design fixes it.** This turned
into a study of the *measurement setup itself* as the tuning knob:

- **More scan positions help** — going from 5 to 7 to 9 antenna positions
  collapses the sideways ambiguity at 60 mm bar spacing.
- **But it is spacing-dependent** — at a tighter 50 mm spacing, even 9 positions
  do not fully resolve it under the current settings.
- **Antenna geometry is the strongest lever** — widening the transmitter–
  receiver offset from 20 mm to 40 mm cleanly separated the tight 50 mm case,
  replicated across three noise patterns.

The deliberate, paper-quality discipline here: we **refused to "fix" the
ambiguity by quietly loosening the confidence threshold.** When the data genuinely
do not support a single sideways position, we report an interval and instead go
find a *physical* lever (more positions, wider antenna offset) that adds real
information.


<img src="../../outputs/experiments/245_coordinate_confidence_aggregate_variable_radius_target2_seed13_21_34/figures/coordinate_confidence_aggregate.png" width="680"><br>
<em><b>Figure.</b> Variable-radius frontier, aggregated across three noise patterns: radius/depth confidence for the hardest (deep, large) bar is strong, while the sideways-position column is reported as an interval until a physical acquisition lever (more positions or wider antenna offset) closes it.</em>

## 8. Honest scorecard — what is solved, and what is still open

**Solved / robust (on 2-D synthetic data):**
- **Position and depth** of bars are recovered accurately and repeatably, even
  with 10% noise, an unknown radar pulse, and several interfering bars.
- **Unknown pulse shape** — once the dominant risk to size — is handled by
  profiling three small pulse nuisances.
- **Detection from scratch** works (48/48 on the benchmark sweep; multiple
  same-depth, variable-depth, and close-spaced bars).
- **Multi-bar coordinate inversion** works on three scene types: same-depth,
  close-spaced, and variable-depth.
- **Uncertainty is reported honestly** — confidence labels and ambiguity
  intervals are mandatory outputs.

**Open / honestly limited:**
- **Size of shallow, thin bars** is an *interval*, not a sharp number, under
  noise + pulse mismatch. Part is fundamental grid rasterization; a
  higher-frequency acquisition is the most promising remedy.
- **Sideways position at very tight spacing** (≤50 mm) can be ambiguous; the
  fix is acquisition design (antenna offset, more positions), not a software
  threshold tweak.
- **Everything is synthetic and 2-D.** No lab or field data yet; 3-D is
  deliberately out of scope for now.

**Tried and deliberately set aside (with evidence, not by assumption):**
the W2 distance as a size objective; blind multi-frequency averaging; free
material parameters in the size optimizer; an "early-reflection" objective; and —
*deferred, not rejected* — the heavy WRI solver and the neural IFWI approach,
both reserved for when explicit nuisance parameters stop being enough (e.g. real
field data).


## 9. Where we are going next

1. **Treat acquisition design as a first-class parameter.** The clearest recent
   wins came from *how we measure* (high-frequency band, antenna offset, number
   of scan positions), not from cleverer post-processing. Formalize a
   **progressive-bandwidth + geometry "recipe"**: low/mid band to find and
   locate, high band and wider antenna offset for final size and sideways
   confidence.

2. **Package the staged pipeline into one reproducible command.** The
   detect ➜ assign ➜ locate ➜ focus ➜ size chain currently spans several
   scripts. Turn it into a single orchestrated runner that always emits the
   confidence labels, intervals, runtimes, and validated figures.

3. **Harden the honest-uncertainty reporting** (calibrate the "weak/moderate/
   strong" thresholds against a real noise model) so the intervals are not just
   conservative but *correct*.

4. **Then step toward realism:** lab or field data, more varied geometries
   (corrosion, coatings, non-circular bars), and — only when explicit nuisance
   parameters genuinely run out — revisit the deferred WRI and neural-IFWI
   branches. The standing goal is a publishable claim: *a source-profiled,
   confidence-aware 2-D GPR-FWI pipeline for rebar location and size with
   explicit ambiguity reporting under noise, pulse mismatch, and multi-bar
   interference.*


## 10. The campaign by the numbers

| Milestone | Plain-English goal | Experiment range | Count* |
|---|---|---|---|
| **M0 — Engine** | Forward simulator + reverse-solver + GPU, recover 3 bars | pre-numbered (`outputs/figures/`) | — |
| **M1 — Baseline** | One bar: nail position + radius on clean data | 001–023 | ~23 |
| **M2 — Five-paper stress test** | Find what really moves radius ➜ source profiling | 024–062 | ~39 |
| **M3 — Multi-bar + confidence** | Several bars; make uncertainty a first-class output | 063–106 | ~44 |
| **M4 — Detection ➜ FWI pipeline** | Find bars from scratch; high-band size confidence | 107–201 | ~95 |
| **M5 — Variable-radius frontier (current)** | Different-sized, close-spaced bars; acquisition design | 202–276 | ~75 |

\*Numbered output folders, approximate per range.

**Category totals** (from the experiment index):

| Category | What it means | Count |
|---|---|---|
| Single rebar | one bar; radius, depth, pulse, bandwidth, material, polish studies | **122** |
| Multi rebar | several bars; coupling, confidence, coordinate optimizer | **80** |
| Infrastructure / reporting | plotting, aggregation, CLI, summary runs (no new physics) | **73** |

Also in the repository: **47** written experiment logs
(`docs/experiments/NN_*.md`), **5** guiding papers
(`docs/papers/`), and on the order of **500+ figures and ~50 animations** across
the experiment folders. Browsing shortcut: a by-category symlink view lives at
`outputs/experiments/_by_category_symlinks/`.


## 11. Glossary — every abbreviation and concept, in one place

- **GPR — Ground-Penetrating Radar.** Surface antenna that pulses radio waves
  into a material and records echoes.
- **Rebar.** Steel reinforcing bar inside concrete. Our targets are circular in
  cross-section; "radius" is half the diameter.
- **B-scan.** The radar image: echo strength vs. antenna position (across) and
  travel-time (down). Buried bars appear as hyperbola (arch) shapes.
- **FDTD — Finite-Difference Time-Domain.** The forward physics engine: simulate
  EM waves by stepping a fine space–time grid via Maxwell's equations.
- **Yee grid / leapfrog.** The standard, textbook arrangement and update order
  for the electric/magnetic fields in FDTD.
- **TMz polarization (Ez, Hx, Hy).** The 2-D field setup we simulate.
- **CPML.** A special absorbing layer at the grid edges so outgoing waves don't
  reflect back — it fakes an infinite medium.
- **FWI — Full-Waveform Inversion.** The reverse-solver: adjust a model until its
  simulated B-scan matches the measured one, fitting the *whole* waveform.
- **Adjoint-state method.** The efficient way to get the model-update direction
  (gradient) with two simulations per step — a forward run plus a backward run of
  the leftover mismatch. Same idea as neural-network backpropagation.
- **Objective / misfit / NRMS.** How wrong the current model is. **NRMS** =
  Normalized Root-Mean-Square error (0 = perfect). Lower is better.
- **Margin.** How clearly the best candidate beats the next-best distinct one.
  Big margin = confident; tiny margin = ambiguous.
- **Ambiguity interval.** The range of values (e.g. radii) the data cannot rule
  out — reported instead of a falsely precise single number.
- **Cycle-skipping.** Classic FWI trap: lining a wave wiggle up with the wrong
  neighbouring wiggle and confidently optimizing toward a wrong model.
- **Optimal transport / Wasserstein (W2).** A "how much work to slide one signal
  onto another" distance; robust to timing shifts, but it discards amplitude
  detail.
- **PEBDD.** Progressively Expanded Bandwidth: fit a narrow low band first, widen
  in stages.
- **WRI.** Wavefield Reconstruction Inversion: a relaxed FWI variant; we borrowed
  its cumulative-frequency idea, not its heavy solver.
- **IFWI.** Implicit FWI: represent the material map with a small neural network;
  deferred here.
- **Source / wavelet profiling ("source-profiled").** Solving for the radar
  pulse's unknown amplitude, start-time, and centre-frequency as small nuisance
  parameters, so a wrong pulse can't corrupt the size answer.
- **Grid rasterization.** Drawing a small circle on a finite grid: two close
  radii can fill identical cells, so the physics literally can't separate them.
- **Guarded revisit / guarded polish.** Going back to rescue a bar that fell into
  the too-thick trap, or a final high-quality local check, after neighbours are
  pinned down.
- **DGX Spark.** The NVIDIA GPU machine this campaign runs on (every simulation
  uses the GPU).

---

*End of note. Every figure above is a direct link into
`outputs/experiments/…/figures/`; open this notebook inside the repository to see
them (animations included).*
